# Clustering Lab

 
Based of the amazing work you did in the Movie Industry you've been recruited to the NBA! You are working as the VP of Analytics that helps support a head scout, Mr. Rooney, for the worst team in the NBA probably the Wizards. Mr. Rooney just heard about Data Science and thinks it can solve all the team's problems!!! He wants you to figure out a way to find players that are high performing but maybe not highly paid that you can steal to get the team to the playoffs! 

In this document you will work through a similar process that we did in class with the NBA data files will be in the canvas assignment, merging them together.

Details: 

- Determine a way to use clustering to estimate based on performance if 
players are under or over paid, generally. 

- Then select players you believe would be best for your team and explain why. Do so in three categories: 
    * Examples that are not good choices (3 or 4) 
    * Several options that are good choices (3 or 4)
    * Several options that could work, assuming you can't get the players in the good category (3 or 4)

- You will decide the cutoffs for each category, so you should be able to explain why you chose them.

- Provide a well commented and clean report of your findings in a separate notebook that can be presented to Mr. Rooney, keeping in mind he doesn't understand...anything. Include a rationale for variables you included in the model, details on your approach and a overview of the results with supporting visualizations. 


Hints:

- Salary is the variable you are trying to understand 
- When interpreting you might want to use graphs that include variables that are the most correlated with Salary
- You'll need to scale the variables before performing the clustering
- Be specific about why you selected the players that you did, more detail is better
- Use good coding practices, comment heavily, indent, don't use for loops unless totally necessary and create modular sections that align with some outcome. If necessary create more than one script,list/load libraries at the top and don't include libraries that aren't used. 
- Be careful for non-traditional characters in the players names, certain graphs won't work when these characters are included.


In [100]:
# Imports
import pandas as pd
import numpy as np
import sklearn as sk

In [101]:
# Load in the data
salary = pd.read_csv("2025_salaries.csv", header=1, encoding='latin-1')
stats = pd.read_csv("nba_2025.txt", sep=",", encoding='latin-1')


In [102]:
# Check to see where to merge on and see the shape of the salary data
print(salary.head())
print(salary.shape)

             Player   Tm    2025-26
0    Jaden Springer  NOP   $70,732 
1  Garrison Mathews  IND  $131,970 
2  Garrison Mathews  IND  $131,970 
3       Mac McClung  IND  $164,060 
4      Didi Louzada  POR  $268,032 
(471, 3)


In [103]:
# Check to see where to merge on and see the shape of the stats data
print(stats.head())
print(stats.shape)

    Rk                   Player   Age Team Pos     G    GS      MP     FG  \
0  1.0  Shai Gilgeous-Alexander  27.0  OKC  PG  49.0  49.0  1632.0  534.0   
1  2.0             Tyrese Maxey  25.0  PHI  PG  52.0  52.0  2008.0  524.0   
2  3.0         Donovan Mitchell  29.0  CLE  SG  51.0  51.0  1719.0  516.0   
3  4.0             Jaylen Brown  29.0  BOS  SF  49.0  49.0  1676.0  534.0   
4  5.0            Luka DonÄiÄ  26.0  LAL  PG  42.0  42.0  1492.0  437.0   

      FGA  ...    TRB    AST    STL   BLK    TOV     PF     PTS  Trp-Dbl  \
0   964.0  ...  218.0  314.0   64.0  38.0  103.0  101.0  1558.0      0.0   
1  1117.0  ...  214.0  351.0  102.0  40.0  126.0  118.0  1503.0      0.0   
2  1060.0  ...  229.0  302.0   79.0  15.0  159.0  125.0  1478.0      0.0   
3  1105.0  ...  336.0  229.0   49.0  20.0  176.0  136.0  1435.0      2.0   
4   923.0  ...  329.0  360.0   61.0  19.0  179.0  102.0  1379.0      6.0   

   Awards  Player-additional  
0     NaN          gilgesh01  
1     NaN         

In [104]:
# Merge the files
merged_data = pd.merge(salary, stats, on='Player')

In [105]:
#Drop variables that will not be needed or are duplicates
duplicates = merged_data[merged_data.duplicated(subset='Player', keep=False)]
print(duplicates)

                     Player   Tm    2025-26     Rk   Age Team Pos     G   GS  \
0          Garrison Mathews  IND  $131,970   398.0  29.0  IND  SG  15.0  1.0   
1          Garrison Mathews  IND  $131,970   398.0  29.0  IND  SG  15.0  1.0   
2               Mac McClung  IND  $164,060   459.0  27.0  2TM  SG   4.0  0.0   
3               Mac McClung  IND  $164,060   459.0  27.0  IND  SG   3.0  0.0   
4               Mac McClung  IND  $164,060   459.0  27.0  CHI  SG   1.0  0.0   
..                      ...  ...        ...    ...   ...  ...  ..   ...  ...   
518  Jeremiah Robinson-Earl  IND        NaN  383.0  25.0  IND  PF  17.0  3.0   
519  Jeremiah Robinson-Earl  IND        NaN  383.0  25.0  DAL  PF   5.0  0.0   
520  Jeremiah Robinson-Earl  IND        NaN  383.0  25.0  2TM  PF  22.0  3.0   
521  Jeremiah Robinson-Earl  IND        NaN  383.0  25.0  IND  PF  17.0  3.0   
522  Jeremiah Robinson-Earl  IND        NaN  383.0  25.0  DAL  PF   5.0  0.0   

        MP  ...    TRB   AST  STL  BLK 

In [106]:
# Drop the duplicate rows but keep the first entry for each player as it is the most recent basketball information
merged_data = merged_data.drop_duplicates(subset='Player', keep='first')
print(merged_data)
#Confirm all duplicates are dropped
print(merged_data.duplicated(subset='Player').sum())

                     Player   Tm       2025-26     Rk   Age Team Pos     G  \
0          Garrison Mathews  IND     $131,970   398.0  29.0  IND  SG  15.0   
2               Mac McClung  IND     $164,060   459.0  27.0  2TM  SG   4.0   
5              Monte Morris  IND     $321,184   470.0  30.0  IND  PG   6.0   
6              E.J. Liddell  PHO     $706,898   461.0  25.0  BRK  PF  10.0   
7             James Wiseman  IND   $1,000,000   480.0  24.0  IND   C   4.0   
..                      ...  ...           ...    ...   ...  ...  ..   ...   
511            Bradley Beal  LAC  $59,020,270   426.0  32.0  LAC  SG   6.0   
513           Stephen Curry  GSW  $59,606,817    24.0  37.0  GSW  PG  39.0   
514          Charles Bassey  MEM           NaN  496.0  25.0  2TM   C   3.0   
517  Jeremiah Robinson-Earl  IND           NaN  383.0  25.0  2TM  PF  22.0   
523        Chris Livingston  MIL           NaN  494.0  22.0  CLE  SF   3.0   

       GS      MP  ...    TRB    AST   STL   BLK    TOV    PF  

In [107]:
# View the column names to determine which variables to keep for clustering 
print(merged_data.columns)

Index(['Player', 'Tm', '2025-26', 'Rk', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP',
       'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%',
       'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV',
       'PF', 'PTS', 'Trp-Dbl', 'Awards', 'Player-additional'],
      dtype='str')


In [108]:
# See if any null values are present in the data
print(merged_data.isnull().sum())


Player                 0
Tm                     0
2025-26                3
Rk                     0
Age                    0
Team                   0
Pos                    0
G                      0
GS                     0
MP                     0
FG                     0
FGA                    0
FG%                    0
3P                     0
3PA                    0
3P%                   20
2P                     0
2PA                    0
2P%                    1
eFG%                   0
FT                     0
FTA                    0
FT%                    3
ORB                    0
DRB                    0
TRB                    0
AST                    0
STL                    0
BLK                    0
TOV                    0
PF                     0
PTS                    0
Trp-Dbl                0
Awards               414
Player-additional      0
dtype: int64


In [109]:
# View the count of the values in the columns '2025-26', '3P%', and 'Awards' with missing values to determine if they are important to keep or not

In [110]:
# View a count of the values in the column '2025-26' 
print(merged_data['2025-26'].value_counts())

2025-26
$2,296,274      32
$1,955,377      16
$2,221,677      15
$1,272,870      10
$2,349,578       5
                ..
$52,627,153      1
$54,708,609      1
$55,224,526      1
$59,020,270      1
$59,606,817      1
Name: count, Length: 285, dtype: int64


In [111]:
# Drop the rows with missing values in the 2025-26 column as it is a small number of rows and I do not think it will be important to keep for clustering
merged_data = merged_data.dropna(subset=['2025-26'])


In [112]:
# Confirm the rows with missing values are dropped
print(merged_data.isnull().sum())

Player                 0
Tm                     0
2025-26                0
Rk                     0
Age                    0
Team                   0
Pos                    0
G                      0
GS                     0
MP                     0
FG                     0
FGA                    0
FG%                    0
3P                     0
3PA                    0
3P%                   19
2P                     0
2PA                    0
2P%                    1
eFG%                   0
FT                     0
FTA                    0
FT%                    3
ORB                    0
DRB                    0
TRB                    0
AST                    0
STL                    0
BLK                    0
TOV                    0
PF                     0
PTS                    0
Trp-Dbl                0
Awards               411
Player-additional      0
dtype: int64


In [113]:
# Do the same with the column '3P%'
print(merged_data['3P%'].value_counts())


3P%
0.000    15
0.333     9
0.321     8
0.350     7
0.250     6
         ..
0.235     1
0.402     1
0.425     1
0.311     1
0.270     1
Name: count, Length: 181, dtype: int64


In [114]:
# I am going to drop this since I do not think 3P% is important becuase like post players do not shoot 3's.
merged_data = merged_data.drop(columns=['3P%'])

In [118]:
# Do the same with the column 'Awards'
print(merged_data['Awards'].value_counts())
# View the range
print(merged_data['Awards'].min())
print(merged_data['Awards'].max())
# Since it is giving me like nan values I am going to replace the nan values with 0 
merged_data['Awards'] = merged_data['Awards'].fillna(0)
# Confirm the nan values are replaced with 0
print(merged_data['Awards'].isnull().sum())
print(merged_data['Awards'].value_counts())
# Re view the range
print(merged_data['Awards'].min())
print(merged_data['Awards'].max())
# Ok so this is not working I am going to try another way
# Ok so after I googled this I had to put inplace=True for it to work
merged_data['Awards'].fillna(0, inplace=True)
# Confirm the nan values are replaced with 0
print(merged_data['Awards'].isnull().sum())

Awards
0.0    411
Name: count, dtype: int64
0.0
0.0
0
Awards
0.0    411
Name: count, dtype: int64
0.0
0.0
0


/tmp/ipykernel_2744/416769121.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  merged_data['Awards'].fillna(0, inplace=True)


In [119]:

# I am going to drop certain variables that I do not think have good correlation to salary or like how good you are at basketball. For example, Team, Pos, Age, GS, MP, and Player-additional.
merged_data = merged_data.drop(columns=['Team', 'Pos', 'Age', 'GS', 'MP', 'Player-additional', 'Tm'])
#Confirm these dropped
print(merged_data.columns)

Index(['Player', '2025-26', 'Rk', 'G', 'FG', 'FGA', 'FG%', '3P', '3PA', '2P',
       '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Trp-Dbl', 'Awards'],
      dtype='str')


In [117]:
# Keep cleaning the data 

In [121]:
# I think Rk, G, FG%, eFG%, TRB, PTS, Trp-Dbl, and Awards are the most important variables to keep for clustering 
# I think FT%, AST, STL, BLK, and PF are not that important but I still want to keep them to see if they have any correlation to salary.
# I am going to drop FG, FGA, 3P, 3PA, 2P, 2PA, 2P%, FT, FTA, ORB, DRB, TOV, and PF becuase I do not think they are important for clustering and I do not want to have too many variables in my clustering. I also do not want to have any variables that are percentages as they are not as important as the other variables and they can be misleading. For example, a player could have a high FG% but if they only take a few shots then it is not as impressive as a player who takes a lot of shots and has a high FG%.
merged_data = merged_data.drop(columns=['FG', 'FGA', '3P', '3PA', '2P', '2PA', '2P%', 'FT', 'FTA', 'ORB',  'DRB', 'TOV', 'PF'])
# Check that I only have left the columns I think are really important and want to see if they are important
print(merged_data.columns)

Index(['Player', '2025-26', 'Rk', 'G', 'FG%', 'eFG%', 'FT%', 'TRB', 'AST',
       'STL', 'BLK', 'PTS', 'Trp-Dbl', 'Awards'],
      dtype='str')


In [ ]:
#Run the clustering algo with your best guess for K

In [125]:
# I am going to do K=5 becuase I hopefully I get like a cluster of the best players, a cluster of the worst players, and then 3 clusters in between. I think this will give me a good idea of how the players are clustered based on their stats and salary.
from sklearn.cluster import KMeans
mymodel = KMeans(n_clusters=3)
mymodel.fit(merged_data[["Rk", "PTS"]])
predictions = mymodel.predict(merged_data[["Rk", "PTS"]])
mymodel.score(merged_data[["Rk", "PTS"]])

-8138263.082930822

In [ ]:
#View the results

In [ ]:
#Create a visualization of the results with 2 or 3 variables that you think will best
#differentiate the clusters

In [ ]:
#Evaluate the quality of the clustering using total variance explained and silhouette scores

In [ ]:
#Determine the ideal number of clusters using the elbow method and the silhouette coefficient

In [ ]:
#Visualize the results of the elbow method

In [ ]:
#Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results

In [ ]:
#Once again evaluate the quality of the clustering using total variance explained and silhouette scores

In [ ]:
#Use the model to select players for Mr. Rooney to consider

In [ ]:
#Write up the results in a separate notebook with supporting visualizations and  an overview of how and why you made the choices you did. This should be at least  500 words and should be written for a non-technical audience.